# 05 — Modal validation against Sinha (2007)

This notebook verifies that the rotor encoded in [`sinha_rotor.toml`](sinha_rotor.toml) reproduces the modal targets reported by Sinha (2007):

| Quantity | Sinha (2007) | Role |
|---|---|---|
| 1st bending, horizontal and vertical | 27.50 Hz | Experimental rap test |
| 1st bending, finite-element | 26.53 Hz | FE model |
| 2nd bending, finite-element | 228.62 Hz | FE model |
| Modal damping, 1st mode | 0.3 % | Stiffness-proportional |
| 1st bending, cracked (fully open) V / H (experimental) | 26.25 Hz | Rap test |
| 1st bending, cracked FE, V / H | 25.75 / 26.10 Hz | FE at crack position |
| 2nd bending, cracked FE, V / H | 226.00 / 226.98 Hz | FE at crack position |

The notebook (1) confirms the healthy baseline, (2) tunes the stiffness-proportional damping coefficient `beta` to match 0.3 %, (3) re-runs the crack case with `crack_model='Gasch'` at `depth_ratio=0.5`, and (4) exports a CSV comparison table.

Scope follows the academic-research skill: every quantity not reported by Sinha is declared in the **Assumptions ledger** below.

## Assumptions ledger

| Parameter | Value | Status | Source / justification |
|---|---|---|---|
| Shaft density `rho` | 7810 kg/m³ | Assumed | ROSS steel default |
| Shaft modulus `E` | 211 GPa | Assumed | ROSS steel default |
| Shaft shear modulus `G_s` | 81.1 GPa | Assumed | ROSS steel default |
| Shaft inner diameter `idl` | 0 (solid) | Assumed | Sinha does not report shaft hollowness |
| Disk mass / inertia | Derived via `DiskElement.from_geometry(scale_factor=3)` | Assumed | Modelling choice, not a Sinha measurement |
| Bearing `kxx = kyy` | Existing 5.45e4 N/m (confirmed vs 27.5 Hz) | Tuned | Fit to Sinha's 27.5 Hz first bending |
| Bearing damping `cxx = cyy` | 50 N·s/m | Assumed | Low arbitrary value, not Sinha data |
| Stiffness-proportional damping `beta` | Tuned (see below) | Tuned | Fit to Sinha's 0.3 % on mode 1 |
| Mass-proportional damping `alpha` | 0 | Assumed | Pure stiffness-proportional damping, per Sinha narrative |
| Crack model | `"Gasch"` | Assumed | Sinha describes a breathing crack; we use the Gasch (1993) formulation |
| Crack fixed-open angle for modal extraction | Evaluate at `ap = 0` and `ap = π/2` | Modelling | Approximates the vertical / horizontal first-bending modes |

In [7]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

if '.' not in sys.path:
    sys.path.insert(0, '.')

import ross as rs
from ross.faults import Crack
import sinha_tools as st

OUT_DIR = Path('study_docs/sinha_validation')
OUT_DIR.mkdir(parents=True, exist_ok=True)

rotor = rs.Rotor.load('sinha_rotor.toml')
print(f'Nodes: {len(rotor.nodes)}, DOFs: {rotor.ndof}')
print(f'Disk node: {rotor.disk_elements[0].n} at x = {rotor.nodes_pos[rotor.disk_elements[0].n]:.3f} m')
print(f'Bearings at nodes: {[b.n for b in rotor.bearing_elements]}')
print(f'Shaft total length: {sum(se.L for se in rotor.shaft_elements):.3f} m')

Nodes: 13, DOFs: 78
Disk node: 6 at x = 0.265 m
Bearings at nodes: [1, 11]
Shaft total length: 0.550 m


## 1. Healthy modal baseline

Run `run_modal` at zero speed and extract the first two bending frequencies. Bending modes are identified by vertical / horizontal translational displacement (DOFs 0 and 1) at the disk node and the absence of axial content.

In [8]:
import scipy.linalg as la

def lateral_disk_amp(mode_vec, disk_node, number_dof):
    mode_vec = np.real(mode_vec)
    i0 = number_dof * disk_node
    return abs(mode_vec[i0]) + abs(mode_vec[i0 + 1])


def pick_two_lateral_bending(freqs_hz, eigvecs, disk_node, number_dof, amp_floor=1e-10, second_min_hz=150.0, ratio=1.42):
    """First two lateral-dominant modes: second must lie above a frequency gap (avoids torsion / local spurious roots)."""
    rows = []
    for i, f in enumerate(freqs_hz):
        if f < 1.0:
            continue
        la = lateral_disk_amp(eigvecs[:, i], disk_node, number_dof)
        if la < amp_floor:
            continue
        rows.append((float(f), i, la))
    rows.sort(key=lambda r: r[0])
    if len(rows) < 2:
        raise RuntimeError("fewer than two lateral candidates")
    f1, i1, _ = rows[0]
    thr = max(second_min_hz, ratio * f1)
    for f, i, la in rows[1:]:
        if f >= thr:
            return [f1, f]
    raise RuntimeError("second lateral bending not found; increase modal extraction bandwidth")


def first_two_bending_modal(modal, disk_node, num_modes_request=72):
    """First two lateral bending frequencies using enough ARPACK modes to reach the second flexural cluster."""
    number_dof = rotor.number_dof
    modal_full = rotor.run_modal(speed=0.0, num_modes=int(num_modes_request), sparse=True)
    wn_hz = modal_full.wn / (2 * np.pi)
    order = np.argsort(wn_hz)
    wn_sorted = wn_hz[order]
    evecs_sorted = modal_full.evectors[:, order]
    n_modes = min(len(wn_sorted), num_modes_request // 2)
    return pick_two_lateral_bending(wn_sorted[:n_modes], evecs_sorted[:, :n_modes], disk_node, number_dof)


modal_h = rotor.run_modal(speed=0.0, num_modes=72, sparse=True)
disk_node = rotor.disk_elements[0].n
bending_h = first_two_bending_modal(modal_h, disk_node=disk_node)
for f_hz in bending_h:
    print(f"Bending mode: {f_hz:.3f} Hz")

f1_healthy_hz = bending_h[0]
f2_healthy_hz = bending_h[1]
print(f"\nFirst bending:  {f1_healthy_hz:.3f} Hz (Sinha FE 26.53, exp 27.50)")
print(f"Second bending: {f2_healthy_hz:.3f} Hz (Sinha FE 228.62; see markdown on FE mesh mismatch)")


Bending mode: 27.500 Hz
Bending mode: 187.606 Hz

First bending:  27.500 Hz (Sinha FE 26.53, exp 27.50)
Second bending: 187.606 Hz (Sinha FE 228.62; see markdown on FE mesh mismatch)


## FE mesh note (healthy second bending)

Sinha (2007) reports a finite-element second bending of 228.62 Hz, whereas the present Timoshenko beam discretisation in ROSS yields approximately 187.6 Hz for the second lateral cluster identified at the disk with a 150 Hz minimum gap between lateral modes (modelling choice to exclude the near-134 Hz branch). This discrepancy is retained in the comparison table as **conflicting evidence** between references: the benchmark FE mesh in Sinha versus the assumed geometry and element layout documented in the assumptions ledger.

The 150 Hz gap threshold is **not** reported by Sinha; it is an independent rule used only to separate the first two flexural lateral branches in this notebook.


## 2. Tune stiffness-proportional damping for 0.3 % on mode 1

Rayleigh damping with `alpha = 0` and `beta != 0` produces a modal damping ratio
\n\\( \zeta_n = \tfrac{1}{2} \beta \omega_n \\),
so to enforce `ζ₁ = 0.003` we set `beta = 2 * 0.003 / ω₁`.

The coefficient is applied element-wise via each `ShaftElement.beta` attribute because `run_modal` consumes the damping matrix the rotor assembly constructs at that stage.

In [9]:
omega1 = 2 * np.pi * f1_healthy_hz
target_zeta = 0.003
beta_target = 2 * target_zeta / omega1
print(f"omega1 = {omega1:.2f} rad/s\nbeta   = {beta_target:.3e}")

for se in rotor.shaft_elements:
    se.beta = beta_target
    se.alpha = 0.0

bearings_zero_damping = [
    rs.BearingElement(n=b.n, kxx=float(np.asarray(b.kxx).reshape(-1)[0]), cxx=0.0)
    for b in rotor.bearing_elements
]
rotor_damped = rs.Rotor(rotor.shaft_elements, rotor.disk_elements, bearings_zero_damping)

zeta1 = 0.5 * beta_target * omega1
print(f"Rayleigh damping on mode 1 (alpha=0): zeta = 0.5*beta*omega1 = {zeta1:.6f} (target 0.0030)")
print("\nNote: bearing dashpots set to cxx=cyy=0 so Rayleigh alpha/beta is the sole damping")
print('source, matching Sinha (2007)\'s "stiffness-proportional damping" description.')


omega1 = 172.79 rad/s
beta   = 3.472e-05
Rayleigh damping on mode 1 (alpha=0): zeta = 0.5*beta*omega1 = 0.003000 (target 0.0030)

Note: bearing dashpots set to cxx=cyy=0 so Rayleigh alpha/beta is the sole damping
source, matching Sinha (2007)'s "stiffness-proportional damping" description.


In [10]:
TOML_OUT = 'sinha_rotor_damped.toml'
rotor_damped.save(TOML_OUT)
print(f'Saved retuned rotor to {TOML_OUT}')

Saved retuned rotor to sinha_rotor_damped.toml


## 3. Cracked modal extraction (Gasch, fully open)

The breathing-crack Gasch model implemented in `ross.faults.crack.Crack` returns the stiffness matrix of the cracked element as a function of the rotation angle `ap`. Evaluating at `ap = 0` and `ap = π/2` yields the stiffness seen when the crack face is aligned with the vertical and horizontal diameters respectively; these configurations serve as proxies for the V / H first-bending frequencies reported by Sinha.

In [11]:
def modal_with_crack_open(rotor, crack_node, depth_ratio, ap):
    """Generalized eigenproblem (K - omega^2 M) q = 0 with cracked element stiffness at fixed crack angle ``ap``."""
    crack = Crack(rotor, n=crack_node, depth_ratio=depth_ratio, crack_model="Gasch")
    K_crack = crack._crack_model(ap)
    dK = K_crack - crack.K_elem
    dof_list = list(crack.dofs)
    K_global = rotor.K(0.0).copy()
    K_global[np.ix_(dof_list, dof_list)] += dK
    M = rotor.M()
    evals, eigvecs = la.eigh(K_global, M)
    omega = np.sqrt(np.maximum(np.real(evals), 0.0))
    freqs_hz = omega / (2 * np.pi)
    number_dof = rotor.number_dof
    disk_node_local = rotor.disk_elements[0].n
    order = np.argsort(freqs_hz)
    fh = freqs_hz[order]
    ev = eigvecs[:, order]
    n_use = min(len(fh), 40)
    return pick_two_lateral_bending(fh[:n_use], ev[:, :n_use], disk_node_local, number_dof)


crack_node = st.SINHA_CRACK_NODE
depth = st.SINHA_CRACK_DEPTH
print(f"Crack at shaft element {crack_node} (x = {rotor_damped.nodes_pos[crack_node]:.3f} m), a/D = {depth}")

freqs_v = modal_with_crack_open(rotor_damped, crack_node, depth, ap=0.0)
freqs_h = modal_with_crack_open(rotor_damped, crack_node, depth, ap=np.pi / 2)
print("\nap = 0 (crack axis horizontal):")
print("  1st bending:", f"{freqs_v[0]:.3f} Hz")
print("  2nd bending:", f"{freqs_v[1]:.3f} Hz")
print("ap = pi/2 (crack axis vertical):")
print("  1st bending:", f"{freqs_h[0]:.3f} Hz")
print("  2nd bending:", f"{freqs_h[1]:.3f} Hz")

cracked_V_1 = min(freqs_v[0], freqs_h[0])
cracked_H_1 = max(freqs_v[0], freqs_h[0])
cracked_V_2 = min(freqs_v[1], freqs_h[1])
cracked_H_2 = max(freqs_v[1], freqs_h[1])
print("\nMapping to Sinha (lowest = vertical):")
print(f"  V 1st bending: {cracked_V_1:.3f} Hz (Sinha FE 25.75)")
print(f"  H 1st bending: {cracked_H_1:.3f} Hz (Sinha FE 26.10)")
print(f"  V 2nd bending: {cracked_V_2:.3f} Hz (Sinha FE 226.00)")
print(f"  H 2nd bending: {cracked_H_2:.3f} Hz (Sinha FE 226.98)")


Crack at shaft element 7 (x = 0.315 m), a/D = 0.5

ap = 0 (crack axis horizontal):
  1st bending: 27.409 Hz
  2nd bending: 184.911 Hz
ap = pi/2 (crack axis vertical):
  1st bending: 24.509 Hz
  2nd bending: 173.772 Hz

Mapping to Sinha (lowest = vertical):
  V 1st bending: 24.509 Hz (Sinha FE 25.75)
  H 1st bending: 27.409 Hz (Sinha FE 26.10)
  V 2nd bending: 173.772 Hz (Sinha FE 226.00)
  H 2nd bending: 184.911 Hz (Sinha FE 226.98)


## 4. Comparison table

Absolute and relative errors are reported against the Sinha (2007) finite-element benchmarks where available, and against the experimental rap-test frequencies where the paper reports them separately. The table is saved to `study_docs/sinha_validation/modal_comparison.csv` for consumption by notebooks 11 and the final validation report.

In [12]:
records = [
    dict(quantity='Healthy 1st bending (FE)',      sinha_hz=26.53,  model_hz=f1_healthy_hz),
    dict(quantity='Healthy 1st bending (exp)',     sinha_hz=27.50,  model_hz=f1_healthy_hz),
    dict(quantity='Healthy 2nd bending (FE)',      sinha_hz=228.62, model_hz=f2_healthy_hz),
    dict(quantity='Cracked 1st bending V (FE)',    sinha_hz=25.75,  model_hz=cracked_V_1),
    dict(quantity='Cracked 1st bending H (FE)',    sinha_hz=26.10,  model_hz=cracked_H_1),
    dict(quantity='Cracked 2nd bending V (FE)',    sinha_hz=226.00, model_hz=cracked_V_2),
    dict(quantity='Cracked 2nd bending H (FE)',    sinha_hz=226.98, model_hz=cracked_H_2),
    dict(quantity='Cracked 1st bending V/H (exp)', sinha_hz=26.25,  model_hz=(cracked_V_1 + cracked_H_1) / 2),
    dict(quantity='Modal damping mode 1',           sinha_hz=target_zeta, model_hz=float(zeta1)),
]
df = pd.DataFrame(records)
df['abs_error'] = df['model_hz'] - df['sinha_hz']
df['rel_error_pct'] = 100 * df['abs_error'] / df['sinha_hz']
df.to_csv(OUT_DIR / 'modal_comparison.csv', index=False)
df

,quantity,sinha_hz,model_hz,abs_error,rel_error_pct
0,Healthy 1st bending (FE),26.530,27.500000,9.700000e-01,3.656238e+00
1,Healthy 1st bending (exp),27.500,27.500000,3.008526e-08,1.094010e-07
2,Healthy 2nd bending (FE),228.620,187.605614,-4.101439e+01,-1.793998e+01
3,Cracked 1st bending V (FE),25.750,24.509304,-1.240696e+00,-4.818237e+00
4,Cracked 1st bending H (FE),26.100,27.408521,1.308521e+00,5.013489e+00
5,Cracked 2nd bending V (FE),226.000,173.771541,-5.222846e+01,-2.310994e+01
6,Cracked 2nd bending H (FE),226.980,184.910640,-4.206936e+01,-1.853439e+01
7,Cracked 1st bending V/H (exp),26.250,25.958912,-2.910878e-01,-1.108906e+00
8,Modal damping mode 1,0.003,0.003000,0.000000e+00,0.000000e+00


## References

Gasch, R. (1993). A survey of the dynamic behaviour of a simple rotating shaft with a transverse crack. *Journal of Sound and Vibration*, 160(2), 313–332.

Sinha, J. K. (2007). Higher order spectra for crack and misalignment identification in the shaft of a rotating machine. *Structural Health Monitoring*, 6(4), 325–334.